In [1]:
import split as sp
import preprocess as pp
import detection as dt
import config as cfg
from pathlib import Path
import utils as ut
import numpy as np

data_folder = Path('/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/')

output_folder = data_folder / 'output'# Create output folder if it doesn't exist
output_folder.mkdir(exist_ok=True)


global_probe, global_probe_data = ut.load_probe_from_json(cfg.PROBE_FILE)
probe_names = 'probe_b'
recording_files = sp.collect_files(data_folder, probe_names, output_folder=False)
shank_dict, recording_paths = sp.split_recording(recording_files, probe_names, global_probe_data)


Found 35 files for probe_b
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
7.59
3596.652
0.0
3589.976
3596.813
3596.793
3596.779
3596.857
3441.519
147.997
0.023
3590.177
208.561
3381.037
3596.843
635.746
1148.36
1795.532
0.0
32.031
31.998
3596.738
3596.531
0.0
Split probe_b into 4 shanks


In [7]:
num = 3.0
shank = shank_dict['probe_b'][num]
shank_folder = output_folder / probe_names[0] / f"shank_{str(num)}"
shank_folder.mkdir(parents=True, exist_ok=True)

In [8]:
shank_probe = ut.load_probe_from_json(cfg.SHANK_FILE)
recording, artifact_indexes = pp.process_traces(shank, shank_probe, n_channels=cfg.N_CHANNELS_SHANK, num_cpus=16, artifacts=False)

46784.553
Processing phase shift...
Bandpass filtering...
Median removal...


In [9]:
import numpy as np
artifact_path = shank_folder / "artifacts"
artifact_path.mkdir(parents=True, exist_ok=True)
np.save(artifact_path / "artifact_indexes.npy", artifact_indexes)

In [10]:
kilosort_params = cfg.KILOSORT_PARAMS
kilosort_params['torch_device'] = 'cuda'
probe, probe_data = ut.load_probe_from_json(cfg.SHANK_FILE)

ks_path = shank_folder / "kilosort"
src_dir = recording_paths
dt.run_kilosort(recording, ks_path, src_dir, kilosort_params)

100%|██████████| 24/24 [05:49<00:00, 14.56s/it]


Sorting Analyzer


AssertionError: gpu is not a valid job keyword argument. Available keyword arguments are: ['n_jobs', 'total_memory', 'chunk_size', 'chunk_memory', 'chunk_duration', 'progress_bar', 'mp_context', 'max_threads_per_process']

In [6]:
import spikeinterface.full as si
params = si.get_default_sorter_params(sorter_name_or_class='kilosort4')
print("Parameters:\n", params)

desc = si.get_sorter_params_description(sorter_name_or_class='kilosort4')
print("Descriptions:\n", desc)

Parameters:
 {'batch_size': 60000, 'nblocks': 1, 'Th_universal': 9, 'Th_learned': 8, 'do_CAR': True, 'invert_sign': False, 'nt': 61, 'shift': None, 'scale': None, 'artifact_threshold': None, 'nskip': 25, 'whitening_range': 32, 'binning_depth': 5, 'sig_interp': 20, 'drift_smoothing': [0.5, 0.5, 0.5], 'nt0min': None, 'dmin': None, 'dminx': 32, 'min_template_size': 10, 'template_sizes': 5, 'nearest_chans': 10, 'nearest_templates': 100, 'max_channel_distance': None, 'templates_from_data': True, 'n_templates': 6, 'n_pcs': 6, 'Th_single_ch': 6, 'acg_threshold': 0.2, 'ccg_threshold': 0.25, 'cluster_downsampling': 20, 'cluster_pcs': 64, 'x_centers': None, 'duplicate_spike_bins': 7, 'do_correction': True, 'keep_good_only': False, 'save_extra_kwargs': False, 'skip_kilosort_preprocessing': False, 'scaleproc': None, 'torch_device': 'auto'}
Descriptions:
 {'batch_size': 'Number of samples included in each batch of data.', 'nblocks': 'Number of non-overlapping blocks for drift correction (additional

In [3]:
import torch
use_cuda = torch.cuda.is_available()
device = "cuda" if use_cuda else "cpu"
print(f"Device: {device}")

Device: cuda


In [8]:
ksp = Path("/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_07_31_npx_test_01/output/probe_b/shank_2.0/kilosort/phy")
amp = np.load(ksp / "amplitudes.npy")


In [2]:
recording_paths

['/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/probe_b_2024-08-06T11_39_02.bin',
 '/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/probe_b_2024-08-06T11_39_28.bin',
 '/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/probe_b_2024-08-06T12_39_35.bin',
 '/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/probe_b_2024-08-06T13_39_29.bin',
 '/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/probe_b_2024-08-06T14_39_30.bin',
 '/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/probe_b_2024-08-06T15_39_30.bin',
 '/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/probe_b_2024-08-06T16_39_30.bin',
 '/groups/voigts/voigtslab/neuropixels_tests_aug_2024/2024_08_06_npx_long_test/data/probe_b_2024-08-06T17_39_31.bin',
 '/groups/voigts/voigtslab/neuropixels_tests_aug_2024/20